# 02 - Regressão logística com regularização reforçada

## Objetivo

Testar `C=0.1` (regularização L2 mais forte) e comparar `class_weight=None` com `class_weight='balanced'`. A seleção usa validação cruzada estratificada de 3 folds apenas na parcela de treino; a divisão fixa de validação continua reservada para a avaliação final. A preparação é sempre ajustada dentro de cada fold pela pipeline comum.

A experiência também inspeciona a fronteira entre `AGE_RANGE` 8 e 9 e as variáveis mais associadas às pontuações dessas classes. A correspondência exata dos códigos a idades e as regras oficiais de aplicação das perguntas precisam de confirmação no dicionário do questionário; associações do modelo não provam causalidade nem leakage.


## Ambiente, dados e divisão fixa

Reutilizamos a divisão estratificada `test_size=0.2`, `random_state=42` da primeira experiência. A versão nova mantém os dados de validação fora da seleção por CV e do ajuste da preparação.


In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, cross_validate

candidate_roots = [Path.cwd(), *Path.cwd().resolve().parents]
repo_root = next(
    (p for p in candidate_roots if (p / 'notebooks').is_dir() and (p / 'src' / 'preparar_dados.py').is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Não foi possível localizar a raiz do projeto com notebooks/ e src/preparar_dados.py.')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.preparar_dados import make_model_pipeline, make_stratified_split, preparar_dados

print(f'Python={sys.version}')
print(f'Raiz do projeto={repo_root}')

train_path = repo_root / 'notebooks' / 'dataset_train.csv'
test_path = repo_root / 'notebooks' / 'dataset_test_X.csv'
if not train_path.is_file() or not test_path.is_file():
    raise FileNotFoundError(f'CSV de treino/teste não encontrados em {repo_root / "notebooks"}')

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
prepared = preparar_dados(train_df, test_df)
X_train, X_val, y_train, y_val = make_stratified_split(
    prepared['X_train'], prepared['y_train'], test_size=0.2, random_state=42
)
print(f'Treino: {X_train.shape}; validação: {X_val.shape}')
print(f'Classes no alvo: {sorted(y_train.unique().tolist())}')


## Comparação por validação cruzada

Comparamos as duas políticas de pesos mantendo `C=0.1` fixo. Os folds são criados exclusivamente dentro de `X_train`/`y_train`; cada ajuste da pipeline volta a aprender imputação e categorias apenas no respetivo fold. `n_jobs=1` limita o consumo de memória. Registamos também avisos de convergência.


In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_results = []

for class_weight in (None, 'balanced'):
    candidate = make_model_pipeline(
        LogisticRegression(
            penalty='l2',
            C=0.1,
            class_weight=class_weight,
            solver='lbfgs',
            max_iter=2000,
            random_state=42,
        ),
        normalize_numeric=True,
    )
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter('always')
        scores = cross_validate(
            candidate,
            X_train,
            y_train,
            cv=cv,
            scoring='f1_macro',
            return_train_score=True,
            n_jobs=1,
            error_score='raise',
        )
    convergence_count = sum(issubclass(w.category, ConvergenceWarning) for w in captured)
    cv_results.append({
        'C': 0.1,
        'class_weight': class_weight,
        'train_macro_f1_mean': scores['train_score'].mean(),
        'validation_macro_f1_mean': scores['test_score'].mean(),
        'validation_macro_f1_std': scores['test_score'].std(),
        'fold_scores': scores['test_score'].tolist(),
        'convergence_warnings': convergence_count,
    })

cv_table = pd.DataFrame(cv_results).sort_values('validation_macro_f1_mean', ascending=False)
display(cv_table)
best_result = cv_table.iloc[0]
best_class_weight = best_result['class_weight']
print(f"Configuração escolhida pela Macro F1 média CV: C=0.1, class_weight={best_class_weight!r}")


## Avaliação na divisão fixa e convergência

A configuração escolhida pela CV é ajustada uma vez em `X_train` e avaliada em `X_val`, sem reajustar a preparação. A avaliação mantém explicitamente as 16 classes. Uma contagem de iterações igual ao limite ou um `ConvergenceWarning` exige investigação antes de aceitar o resultado.


In [ ]:
selected_params = {
    'penalty': 'l2',
    'C': 0.1,
    'class_weight': best_class_weight,
    'solver': 'lbfgs',
    'max_iter': 2000,
    'random_state': 42,
    'normalize_numeric': True,
}

validation_pipe = make_model_pipeline(
    LogisticRegression(**{k: selected_params[k] for k in ('penalty', 'C', 'class_weight', 'solver', 'max_iter', 'random_state')}),
    normalize_numeric=selected_params['normalize_numeric'],
)
with warnings.catch_warnings(record=True) as captured:
    warnings.simplefilter('always')
    validation_pipe.fit(X_train, y_train)
validation_convergence = [w for w in captured if issubclass(w.category, ConvergenceWarning)]
model = validation_pipe.named_steps['model']
print(f'Avisos de convergência: {len(validation_convergence)}')
print(f'Iterações por ajuste: {model.n_iter_.tolist()} / limite {selected_params["max_iter"]}')

y_train_pred = validation_pipe.predict(X_train)
y_val_pred = validation_pipe.predict(X_val)
print(f'Macro F1 treino: {f1_score(y_train, y_train_pred, average="macro"):.4f}')
print(f'Macro F1 validação: {f1_score(y_val, y_val_pred, average="macro"):.4f}')
print(classification_report(
    y_val, y_val_pred, labels=list(range(16)), zero_division=0,
    target_names=[str(i) for i in range(16)],
))
confusion = confusion_matrix(y_val, y_val_pred, labels=list(range(16)))
display(pd.DataFrame(confusion, index=range(16), columns=range(16)))


## Diagnóstico da fronteira entre as classes 8 e 9

A matriz de confusão mostra os erros entre estas classes adjacentes. Para procurar pistas de perguntas condicionadas, inspecionamos os maiores contrastes dos coeficientes entre os logits das classes 8 e 9 e as respostas observadas nas classes 7–10. Estes resultados são associações preditivas: a confirmação de uma regra etária requer o dicionário do questionário e a definição oficial de `AGE_RANGE`.


In [ ]:
class_labels = model.classes_
if 8 in class_labels and 9 in class_labels:
    i8 = int(np.flatnonzero(class_labels == 8)[0])
    i9 = int(np.flatnonzero(class_labels == 9)[0])
    feature_names = np.asarray(validation_pipe.named_steps['preprocessor'].output_columns_)
    contrast = model.coef_[i9] - model.coef_[i8]
    top_idx = np.argsort(np.abs(contrast))[-20:][::-1]
    display(pd.DataFrame({
        'característica': feature_names[top_idx],
        'coeficiente_logit_9_menos_8': contrast[top_idx],
    }))

raw_train = train_df.loc[y_train.index]
for column in ('IN22', 'PC1B', 'HA1A', 'IN15', 'IN16'):
    if column not in raw_train.columns:
        continue
    print(f'\n{column}: proporções por classe')
    subset = raw_train.loc[y_train.isin([7, 8, 9, 10]), [column]].copy()
    subset['AGE_RANGE'] = y_train.loc[subset.index]
    proportions = pd.crosstab(
        subset['AGE_RANGE'], subset[column].fillna('<ausente>'), normalize='index'
    )
    display(proportions.round(3))

print('Erros verdadeiros 8 previstos como 9:', int(confusion[8, 9]))
print('Erros verdadeiros 9 previstos como 8:', int(confusion[9, 8]))


## Gerar a segunda submissão

Criamos e ajustamos uma pipeline nova no treino completo. O `RowId` é copiado do ficheiro de exemplo, quando disponível, ou da submissão anterior já validada; nunca são usadas previsões do exemplo. O ficheiro novo não substitui submissões existentes.


In [ ]:
final_pipe = make_model_pipeline(
    LogisticRegression(**{k: selected_params[k] for k in ('penalty', 'C', 'class_weight', 'solver', 'max_iter', 'random_state')}),
    normalize_numeric=selected_params['normalize_numeric'],
)
with warnings.catch_warnings(record=True) as captured:
    warnings.simplefilter('always')
    final_pipe.fit(prepared['X_train'], prepared['y_train'])
final_convergence = [w for w in captured if issubclass(w.category, ConvergenceWarning)]
print(f'Avisos de convergência no ajuste final: {len(final_convergence)}')
print(f'Iterações do ajuste final: {final_pipe.named_steps["model"].n_iter_.tolist()}')
test_predictions = final_pipe.predict(prepared['X_test']).astype(int)

example_path = repo_root / 'example_submission_10.csv'
previous_path = repo_root / 'submissions' / 'submission_001_regressao_logistica.csv'
if example_path.is_file():
    row_reference = pd.read_csv(example_path, usecols=['RowId'])
    reference_source = example_path
elif previous_path.is_file():
    row_reference = pd.read_csv(previous_path, usecols=['RowId'])
    reference_source = previous_path
elif 'RowId' in test_df.columns:
    row_reference = test_df[['RowId']].copy()
    reference_source = 'dataset_test_X.csv:RowId'
else:
    raise FileNotFoundError('É necessário example_submission_10.csv, a submissão anterior validada ou RowId no teste.')

assert len(test_predictions) == len(row_reference) == 3655
submission = row_reference.copy()
submission['AGE_RANGE'] = test_predictions
assert list(submission.columns) == ['RowId', 'AGE_RANGE']
assert submission['RowId'].equals(row_reference['RowId'])
assert pd.api.types.is_integer_dtype(submission['AGE_RANGE'])
assert submission['AGE_RANGE'].between(0, 15).all()

submission_dir = repo_root / 'submissions'
submission_dir.mkdir(exist_ok=True, parents=True)
submission_path = submission_dir / 'submission_002_regressao_logistica_regularizada.csv'
suffix = 2
while submission_path.exists():
    suffix += 1
    submission_path = submission_dir / f'submission_002_regressao_logistica_regularizada_{suffix}.csv'
submission.to_csv(submission_path, index=False)
saved = pd.read_csv(submission_path)
assert saved.shape == (3655, 2)
assert list(saved.columns) == ['RowId', 'AGE_RANGE']
assert saved['RowId'].equals(row_reference['RowId'])
assert saved['AGE_RANGE'].between(0, 15).all()
print(f'Submissão guardada em {submission_path}')
print(f'RowId copiado de {reference_source}')
print(saved['AGE_RANGE'].value_counts().sort_index())


## Registo e conclusão desta execução

Nesta execução local, `C=0.1` sem ponderação teve Macro F1 média de validação cruzada **0.4246** (desvio padrão **0.0059**); `class_weight='balanced'` obteve **0.4186** (desvio padrão **0.0042**). A configuração sem ponderação foi escolhida. Na divisão fixa, obteve Macro F1 de treino **0.7597** e validação **0.4125**, sem avisos de convergência (`n_iter=260`). A avaliação de validação melhorou face à primeira experiência (**0.3937**), e a diferença treino-validação diminuiu, mas permanece relevante.

Na matriz de confusão, não houve previsões cruzadas entre as classes 8 e 9 (0 erros 8→9 e 0 erros 9→8). Na inspeção dos dados de treino, `IN22` e as categorias `Não aplicável` de perguntas como `PC1B`, `HA1A` e `IN15` aparecem com contrastes fortes: `IN22='Não aplicável'` ocorre em cerca de 57,6% da classe 8 e 91,6% da classe 9; `IN22='Está na menopausa'`, em 20,7% e 4,2%, respetivamente. Isto é evidência de associação preditiva e uma pista de perguntas condicionadas/skip patterns, mas não confirma que a fronteira seja exatamente aos 55 anos. A associação também pode codificar diferenças de sexo, comportamento ou elegibilidade. A correspondência entre os códigos de `AGE_RANGE` e idades não está documentada nos ficheiros locais; confirmar com o codebook antes de excluir variáveis ou alterar a preparação.

Nota de comparabilidade: a primeira experiência usava `solver='saga'`; esta usa `solver='lbfgs'`, que também aceita a representação esparsa. Por isso, a melhoria face ao primeiro resultado não pode ser atribuída apenas à redução de `C` ou à política de pesos.

- Data: 08/11/2026
- Pontuação pública no Kaggle: 0.44
- Conclusão: A regularização reforçada melhorou a validação e o Kaggle; a diferença treino–validação diminuiu, mas continua elevada; class_weight="balanced" teve pior resultado na validação cruzada; o solver também mudou, pelo que a melhoria não pode ser atribuída apenas à regularização.
- Melhorias: Testar C=0.01, 0.03 e 0.3, mantendo o mesmo solver; investigar as perguntas condicionadas pela idade.

